# Week 8 — Wednesday Lab Notebook  
## Regression

### Today’s Goal

By the end of today, students should be able to:

- understand what **regression** means in machine learning
- know the difference between **regression** and **classification**
- build a simple **Linear Regression** model
- understand **features** and a numeric **target**
- make predictions using a trained regression model
- evaluate a regression model using **MAE**, **MSE**, **RMSE**, and **R²**
- understand **underfitting** and **overfitting** in simple words
- understand the basic idea of **regularization**
- know when **Ridge** and **Lasso** are used conceptually

## 3-Hour Structure

**0:00–0:20** What regression is and where it is used  
**0:20–0:45** Build and inspect a small house-price dataset  
**0:45–1:15** Train a Linear Regression model  
**1:15–1:40** Predictions and regression metrics  
**1:40–2:05** Single-feature line intuition  
**2:05–2:30** Underfitting vs overfitting  
**2:30–2:50** Regularization idea: Ridge and Lasso  
**2:50–3:00** Practice + wrap-up

## Part A — What is Regression?

Regression is used when the output we want to predict is a **number**.

Examples:

- house price
- salary
- marks
- monthly sales
- temperature
- delivery time

So if the answer is numeric, regression is often the right choice.

## Part B — Regression vs Classification

### Regression
Predicts a number.

Examples:
- predict house price = `8500000`
- predict salary = `75000`
- predict exam marks = `88`

### Classification
Predicts a category or label.

Examples:
- pass / fail
- spam / not spam
- approved / rejected

So today's topic is about **numeric prediction**.

## Part C — Real-Life Use Cases

Regression is used in many real situations:

1. **Education**  
   Predict student marks based on study hours, attendance, and assignments.

2. **Business**  
   Predict monthly sales from advertisement budget and season.

3. **Real Estate**  
   Predict house price from area, number of rooms, and location.

4. **Health**  
   Predict hospital stay duration or treatment cost.

5. **Agriculture**  
   Predict crop production based on land area, rainfall, and fertilizer.

## Part D — Import Libraries

We will use:

- `pandas` and `numpy` for data
- `train_test_split` to split data
- `LinearRegression` as our main regression model
- `mean_absolute_error`, `mean_squared_error`, and `r2_score` for evaluation
- `matplotlib` to understand patterns visually
- `PolynomialFeatures`, `Ridge`, and `Lasso` for concept-level advanced practice

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

## Part E — Create a Small House Price Dataset

We will create a simple dataset.

### Goal
Predict the **price** of a house.

### Features
- `area_sqft`
- `bedrooms`
- `house_age`
- `distance_city_km`

### Target
- `price_lakh`

`price_lakh` means house price in **lakhs**.

In [ ]:
houses = pd.DataFrame({
    "area_sqft": [800, 950, 1100, 1200, 1400, 1500, 1600, 1750, 1800, 2000, 2100, 2300, 900, 1300, 1700, 1900, 2200, 2500, 1450, 1550],
    "bedrooms": [2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 2, 3, 4, 4, 5, 5, 3, 4],
    "house_age": [15, 12, 10, 8, 7, 6, 5, 4, 8, 3, 2, 1, 14, 9, 5, 4, 2, 1, 7, 6],
    "distance_city_km": [14, 12, 10, 9, 8, 7, 6, 5, 7, 4, 4, 3, 13, 9, 6, 5, 4, 3, 8, 7],
    "price_lakh": [42, 47, 55, 60, 68, 74, 78, 85, 82, 96, 102, 110, 45, 62, 84, 92, 105, 118, 70, 76]
})

houses

## Part F — First Look at the Data

Before training any model, always inspect the dataset.

In [ ]:
print(houses.head())
print()
print(houses.info())

In [ ]:
houses.describe()

### What should students notice?

- all feature columns are numeric
- target column is also numeric
- larger area often seems linked with higher price
- smaller distance to city often seems linked with higher price
- newer houses often seem more expensive

## Part G — Separate Features and Target

In machine learning:

- `X` = input features
- `y` = output target

In [ ]:
X = houses[["area_sqft", "bedrooms", "house_age", "distance_city_km"]]
y = houses["price_lakh"]

print("Features:")
print(X.head())
print()
print("Target:")
print(y.head())

## Part H — Split Data into Train and Test

We train the model on one part of the data and test it on unseen data.

This helps us check whether the model learned properly.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Part I — What is Linear Regression?

Linear Regression tries to find a relationship like this:

**price = m1(feature1) + m2(feature2) + m3(feature3) + ... + c**

That means the model learns:

- how much each feature affects the target
- one constant starting value called the **intercept**

It is called **linear** because the relationship is modeled in a straight-line form.

## Part J — Train a Linear Regression Model

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Model training complete.")

## Part K — Intercept and Coefficients

After training, the model learns:

- **intercept**
- **coefficients** for each feature

A positive coefficient means:

> when that feature increases, the predicted target also tends to increase.

A negative coefficient means:

> when that feature increases, the predicted target tends to decrease.

In [ ]:
print("Intercept:", lr_model.intercept_)
print()

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr_model.coef_
})

coef_table

### How to read this table?

If the coefficient of `area_sqft` is positive, then larger area increases predicted price.

If the coefficient of `distance_city_km` is negative, then larger distance from city decreases predicted price.

This is the basic interpretation idea.

## Part L — Make Predictions on Test Data

In [ ]:
y_pred = lr_model.predict(X_test)

comparison = X_test.copy()
comparison["actual_price_lakh"] = y_test.values
comparison["predicted_price_lakh"] = np.round(y_pred, 2)
comparison["error"] = np.round(comparison["actual_price_lakh"] - comparison["predicted_price_lakh"], 2)

comparison

## Part M — Regression Metrics in Simple Words

We need to measure how good our predictions are.

### 1. MAE — Mean Absolute Error
Average absolute difference between actual and predicted values.

### 2. MSE — Mean Squared Error
Average of squared errors.

### 3. RMSE — Root Mean Squared Error
Square root of MSE.  
This is easier to understand because it comes back to the original unit.

### 4. R² Score
Shows how much variance in the target is explained by the model.

General idea:
- closer to `1` is better
- very low value means weak model

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 3))
print("MSE :", round(mse, 3))
print("RMSE:", round(rmse, 3))
print("R^2 :", round(r2, 3))

## Part N — Actual vs Predicted Values Plot

A good regression model should produce predictions close to the actual values.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Price (lakh)")
plt.ylabel("Predicted Price (lakh)")
plt.title("Actual vs Predicted House Prices")

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.show()

### What should students understand from this graph?

- if points are close to the diagonal line, predictions are good
- if points are far away from the line, prediction errors are larger

## Part O — Single-Feature Line Intuition

Now let us use only one feature: `area_sqft`.

This is not always the best real model, but it helps students see the line idea clearly.

In [ ]:
X_one = houses[["area_sqft"]]
y_one = houses["price_lakh"]

one_feature_model = LinearRegression()
one_feature_model.fit(X_one, y_one)

print("Intercept:", round(one_feature_model.intercept_, 3))
print("Coefficient for area_sqft:", round(one_feature_model.coef_[0], 5))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(houses["area_sqft"], houses["price_lakh"])

x_line = pd.DataFrame({
    "area_sqft": np.linspace(houses["area_sqft"].min(), houses["area_sqft"].max(), 100)
})
y_line = one_feature_model.predict(x_line)

plt.plot(x_line["area_sqft"], y_line)
plt.xlabel("Area (sqft)")
plt.ylabel("Price (lakh)")
plt.title("Simple Linear Regression Line")
plt.show()

### Teaching point

This line is the model’s best straight-line estimate.

It does **not** pass through every point exactly.

Instead, it tries to stay as close as possible to the full trend.

## Part P — Residuals in Simple Words

A **residual** means:

**actual value - predicted value**

If residual is small, prediction is close.

If residual is large, prediction is far from the actual value.

In [ ]:
comparison["residual"] = np.round(comparison["actual_price_lakh"] - comparison["predicted_price_lakh"], 2)
comparison[["actual_price_lakh", "predicted_price_lakh", "residual"]]

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(comparison["predicted_price_lakh"], comparison["residual"])
plt.axhline(0)
plt.xlabel("Predicted Price (lakh)")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

### What should students notice?

- points near zero mean small error
- very large positive or negative residual means larger mistake
- residual plots help us see error behavior

## Part Q — Underfitting and Overfitting

### Underfitting
The model is too simple.  
It cannot learn the pattern well.

### Overfitting
The model learns training data too closely.  
It performs well on training data but badly on new data.

Good ML means:
> learn the real pattern, not just memorize the training examples.

## Part R — Create a Small Curved Dataset for Understanding Fit Quality

In [ ]:
np.random.seed(42)

x_curve = np.linspace(0, 10, 40)
y_curve = 3 + 2*x_curve + 0.8*(x_curve**2) + np.random.normal(0, 8, size=40)

curve_df = pd.DataFrame({"x": x_curve, "y": y_curve})
curve_df.head()

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    curve_df[["x"]], curve_df["y"], test_size=0.25, random_state=42
)

## Part S — Compare Different Polynomial Degrees

We will compare:

- degree 1 → very simple
- degree 2 → often better for curved data
- degree 10 → may become too flexible

This is a simple way to understand underfitting and overfitting.

In [ ]:
results = []

for degree in [1, 2, 10]:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("lr", LinearRegression())
    ])

    model.fit(Xc_train, yc_train)

    train_pred = model.predict(Xc_train)
    test_pred = model.predict(Xc_test)

    train_rmse = mean_squared_error(yc_train, train_pred) ** 0.5
    test_rmse = mean_squared_error(yc_test, test_pred) ** 0.5

    results.append({
        "degree": degree,
        "train_rmse": round(train_rmse, 3),
        "test_rmse": round(test_rmse, 3)
    })

pd.DataFrame(results)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(curve_df["x"], curve_df["y"], label="Data")

x_plot = pd.DataFrame({
    "x": np.linspace(curve_df["x"].min(), curve_df["x"].max(), 200)
})

for degree in [1, 2, 10]:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("lr", LinearRegression())
    ])
    model.fit(Xc_train, yc_train)
    y_plot = model.predict(x_plot)
    plt.plot(x_plot["x"], y_plot, label=f"Degree {degree}")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Underfitting vs Better Fit vs Possible Overfitting")
plt.legend()
plt.show()

### How to explain this in class?

- **Degree 1** may miss the curve → underfitting  
- **Degree 2** may capture the main pattern better  
- **Degree 10** may become too flexible and memorize noise  

The exact results depend on the dataset, but the concept is the key lesson.

## Part T — Regularization Idea in Simple Words

Sometimes a model becomes too complex.

Regularization is a method to:

- control model complexity
- reduce extreme coefficients
- help generalization on unseen data

Two common names:

### Ridge
Pushes coefficients smaller, but usually does not make them exactly zero.

### Lasso
Can shrink some coefficients all the way to zero.  
This can behave a little like feature selection.

## Part U — Small Ridge and Lasso Example

We will use polynomial features again and compare:

- Linear Regression
- Ridge
- Lasso

This is only for concept understanding.

In [ ]:
X_curve_full = curve_df[["x"]]
y_curve_full = curve_df["y"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_curve_full, y_curve_full, test_size=0.25, random_state=42
)

linear_poly = Pipeline([
    ("poly", PolynomialFeatures(degree=10, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ridge_poly = Pipeline([
    ("poly", PolynomialFeatures(degree=10, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

lasso_poly = Pipeline([
    ("poly", PolynomialFeatures(degree=10, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.1, max_iter=10000))
])

models = {
    "LinearRegression": linear_poly,
    "Ridge": ridge_poly,
    "Lasso": lasso_poly
}

reg_results = []

for name, model in models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    rmse = mean_squared_error(yr_test, pred) ** 0.5
    r2 = r2_score(yr_test, pred)
    reg_results.append({
        "model": name,
        "test_rmse": round(rmse, 3),
        "r2_score": round(r2, 3)
    })

pd.DataFrame(reg_results)

### Beginner takeaway

You do not need to memorize all math today.

Just remember:

- **Linear Regression** is the basic starting point
- **Ridge** and **Lasso** are useful when we want to control complexity
- regularization helps when models become too flexible

## Part V — Important Terms for Students

### Feature
Input column used for prediction

### Target
Output column we want to predict

### Model
Learns the relationship between inputs and output

### Training
Learning from data

### Prediction
Using the learned model on new data

### Error
Difference between actual and predicted value

## Part W — Common Beginner Mistakes

1. using regression when the target is actually a category  
2. forgetting to separate features and target  
3. training on all data without testing on unseen data  
4. thinking low training error always means good model  
5. not checking units of the target  
6. confusing MAE, MSE, and RMSE  
7. trying a very complex model too early  
8. ignoring overfitting and underfitting

## Part X — Small Real-Life Example 1: Student Marks Prediction

Possible features:
- study hours
- attendance
- previous marks
- assignment completion

Target:
- final exam marks

This is regression because final marks are numeric.

## Part Y — Small Real-Life Example 2: Sales Prediction

Possible features:
- advertising budget
- shop visitors
- discount percentage
- season score

Target:
- monthly sales

Again, this is numeric prediction.

## Part Z — Small Real-Life Example 3: House Rent Prediction

Possible features:
- area
- rooms
- location score
- age of house
- furnished or not

Target:
- monthly rent

This is another common regression problem.

## Part AA — Mini In-Class Practice

Try these before moving to the after-lab tasks:

1. create `X` and `y` from the house dataset  
2. split data into train and test sets  
3. train a `LinearRegression()` model  
4. print the intercept  
5. print the coefficients  
6. make predictions on test data  
7. create a comparison table of actual and predicted values  
8. calculate `MAE`, `MSE`, `RMSE`, and `R²`  
9. explain whether the result looks good or weak  
10. write in one line whether this problem is regression or classification

## Part AB — Recap

Today we learned how to:

- understand regression in simple words
- identify numeric prediction problems
- build a Linear Regression model
- interpret intercept and coefficients
- make predictions
- evaluate using MAE, MSE, RMSE, and R²
- understand residuals
- understand underfitting and overfitting
- understand the basic idea of Ridge and Lasso

# After-Lab Tasks (10)

### Task 1
Create a small dataset with at least 15 rows for one numeric prediction problem such as house price, salary, marks, or sales.

### Task 2
From your dataset, separate the feature columns and the target column.

### Task 3
Split your data into training and test sets using `train_test_split()`.

### Task 4
Train a `LinearRegression()` model on the training data.

### Task 5
Print the **intercept** and **coefficients** of your trained model.

### Task 6
Predict on the test set and make a table with:
- actual values
- predicted values
- error

### Task 7
Calculate:
- MAE
- MSE
- RMSE
- R²

### Task 8
Use only **one feature** from your dataset and draw a scatter plot with the regression line.

### Task 9
Write in your own words:
- what underfitting means
- what overfitting means

### Task 10
Write a short note of 4 to 6 lines on:
- Linear Regression
- Ridge
- Lasso
and explain their difference in simple wording.

# Optional Homework Challenge

Create your own full regression mini-project.

Choose one problem:
- student marks prediction
- monthly sales prediction
- employee salary prediction
- house rent prediction

Then do all of the following:

1. explain the problem in 2 to 3 lines  
2. create at least 20 rows of data  
3. choose at least 3 feature columns  
4. choose 1 numeric target column  
5. split data into train and test  
6. train a `LinearRegression()` model  
7. evaluate using `MAE`, `MSE`, `RMSE`, and `R²`  
8. show a comparison table of actual vs predicted values  
9. write one paragraph on whether the model seems good or weak  
10. write one paragraph on how overfitting could happen in this problem